# 1. Reranking Baseline: Personalised Content Similarity

A personalised baseline with no learned weights. Each candidate is scored by its cosine
similarity to the user's **causal clicked-history profile**: the mean of the article
vectors the user clicked before the impression.

Five rules are compared on validation:

- **Text only**: cosine between candidate and user text profiles
- **Entity only**: cosine between candidate and user entity profiles
- **Fusion** with text weight $\alpha \in \{0.25, 0.50, 0.75\}$: $\alpha\, s^{text} + (1-\alpha)\, s^{entity}$

The rule with the best validation nDCG@10 is the only one evaluated on test.

## 1.1 Load Packages

In [3]:
import os
import sys
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from IPython.display import display

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/submission/4.Code")
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()
else:
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    while (
        not (PROJECT_ROOT / "5_Models").exists() and PROJECT_ROOT != PROJECT_ROOT.parent
    ):
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [5]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = min((RUN / f"ranking_features_{split}").glob("*.parquet"))
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

,impression_uid,news_code,user_code,article_row,timestamp,candidate_source,eligible,category_code,subcategory_code,title_word_count,...,multi_interest_match_score,interest_group_count,selected_interest_group_count,multi_interest_valid,hours_since_previous_impression,first_visit,text_match_valid,entity_match_valid,long_term_match_valid,recent_match_valid
0,20112,1972,32894,1542,2019-11-09 00:00:19,logged,True,14.0,230.0,9.0,...,0.271657,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0
1,20112,2678,32894,2072,2019-11-09 00:00:19,logged,True,13.0,95.0,17.0,...,0.501291,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0
2,20112,5379,32894,4188,2019-11-09 00:00:19,logged,True,14.0,231.0,22.0,...,0.348739,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0
3,20112,6868,32894,5378,2019-11-09 00:00:19,logged,True,7.0,118.0,15.0,...,0.324235,3.0,3.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0
4,20112,7938,32894,6200,2019-11-09 00:00:19,logged,True,2.0,61.0,11.0,...,0.162289,3.0,3.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0


In [6]:
validation_preview.head()

,impression_uid,news_code,user_code,article_row,timestamp,candidate_source,eligible,category_code,subcategory_code,title_word_count,...,multi_interest_match_score,interest_group_count,selected_interest_group_count,multi_interest_valid,hours_since_previous_impression,first_visit,text_match_valid,entity_match_valid,long_term_match_valid,recent_match_valid
0,10154218,12029,11484,9378,2019-11-13 00:01:00,logged,True,10.0,160.0,18.0,...,0.550584,3.0,3.0,1.0,85.836945,0.0,1.0,1.0,1.0,0.0
1,10154218,47020,11484,36921,2019-11-13 00:01:00,logged,True,11.0,175.0,4.0,...,0.718951,3.0,3.0,1.0,85.836945,0.0,1.0,0.0,1.0,0.0
2,10135780,103,6069,88,2019-11-13 00:01:23,logged,True,10.0,51.0,10.0,...,0.301457,3.0,3.0,1.0,5.919445,0.0,1.0,1.0,1.0,1.0
3,10135780,261,6069,211,2019-11-13 00:01:23,logged,True,10.0,160.0,7.0,...,0.277886,3.0,3.0,1.0,5.919445,0.0,1.0,0.0,1.0,1.0
4,10135780,1644,6069,1293,2019-11-13 00:01:23,logged,True,14.0,232.0,10.0,...,0.592967,3.0,3.0,1.0,5.919445,0.0,1.0,1.0,1.0,1.0


# 2. Model: Causal Profile Matching

Feature engineering mean-pools each user's earlier clicked article vectors, separately
for text and entities, and stores the candidate–profile cosines with validity flags.
The scoring rule handles missing data explicitly:

- If both modalities are valid, fusion blends them; otherwise it uses whichever is valid
- If the user has no matching profile at all, the rule falls back to causal popularity
- A candidate with no valid vector scores −2, below every valid cosine in [−1, 1]

```text
Earlier clicked articles → text and entity user profiles
Candidate + each profile → cosine similarity → text / entity / blended score
```

In [7]:
CONTENT_COLUMNS = [
    "smoothed_prior_article_ctr",
    "text_cosine",
    "entity_cosine",
    "text_match_valid",
    "entity_match_valid",
    "no_text_profile",
    "no_entity_profile",
]
MISSING_SCORE = -2.0  # strictly below the valid cosine range [-1, 1]


def content_score(frame, trial):
    """Score candidates with one text, entity or fusion rule."""
    prior = frame["smoothed_prior_article_ctr"].to_numpy(dtype=float)
    text = frame["text_cosine"].to_numpy(dtype=float)
    entity = frame["entity_cosine"].to_numpy(dtype=float)
    text_valid = frame["text_match_valid"].to_numpy(dtype=bool)
    entity_valid = frame["entity_match_valid"].to_numpy(dtype=bool)
    no_text_profile = frame["no_text_profile"].to_numpy(dtype=bool)
    no_entity_profile = frame["no_entity_profile"].to_numpy(dtype=bool)

    if trial["kind"] == "text":
        return np.where(
            no_text_profile, prior, np.where(text_valid, text, MISSING_SCORE)
        )
    if trial["kind"] == "entity":
        return np.where(
            no_entity_profile, prior, np.where(entity_valid, entity, MISSING_SCORE)
        )

    # Fusion: blend when both are valid, otherwise use the valid modality
    alpha = trial["text_weight"]
    blended = np.where(
        text_valid & entity_valid,
        alpha * text + (1 - alpha) * entity,
        np.where(text_valid, text, np.where(entity_valid, entity, MISSING_SCORE)),
    )
    return np.where(no_text_profile & no_entity_profile, prior, blended)

# 3. Experiment Configuration

`alpha` (`text_weight`) is the text share in the fusion rules; entity receives `1 - alpha`.

In [8]:
SELECTION_METRIC = "ndcg_at_10"
OUTPUT_DIR = ARTIFACTS / "content_similarity"
POPULARITY_MODEL = "popularity_smoothed_prior_article_ctr"

TRIALS = {
    "content_text_cosine": {"kind": "text"},
    "content_entity_cosine": {"kind": "entity"},
    "content_fusion_text_0.25": {"kind": "fusion", "text_weight": 0.25},
    "content_fusion_text_0.50": {"kind": "fusion", "text_weight": 0.50},
    "content_fusion_text_0.75": {"kind": "fusion", "text_weight": 0.75},
}

# 4. Validation Comparison

Every rule is scored on the complete validation slates. If the popularity notebook has
been run, its validation result is added as context and as the baseline for
observed-CTR lift; it does not take part in selection.

In [9]:
def evaluate_trial(name, split):
    """Score one content rule on a split and evaluate it."""
    result, _ = score_and_evaluate(
        lambda frame: content_score(frame, TRIALS[name]),
        split=split,
        feature_run=RUN,
        output_dir=OUTPUT_DIR / name,
        model_name=name,
        columns=CONTENT_COLUMNS,
        results_dir=RESULTS,
    )
    return result


validation_results = [evaluate_trial(name, "validation") for name in TRIALS]
leaderboard = compare_results(validation_results, sort_by=SELECTION_METRIC)
final_model_name = leaderboard["model"][0]
print(f"Validation-selected content rule: {final_model_name}")

popularity_path = RESULTS / f"reranking_{POPULARITY_MODEL}_validation.csv"
if popularity_path.exists():
    validation_results.append(popularity_path)
comparison = compare_results(
    validation_results, baseline_model=POPULARITY_MODEL, sort_by=SELECTION_METRIC
)
comparison.write_csv(RESULTS / "reranking_content_similarity_comparison_validation.csv")
display(comparison.select("model", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

Validation-selected content rule: content_fusion_text_0.50


model,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,f64,f64,f64,f64
"""content_fusion_text_0.50""",0.307498,0.287405,0.343959,0.569667
"""content_fusion_text_0.75""",0.304778,0.285018,0.341506,0.568212
"""content_fusion_text_0.25""",0.299618,0.27887,0.336417,0.563503
"""content_text_cosine""",0.288871,0.268179,0.325204,0.551759
"""content_entity_cosine""",0.276967,0.255999,0.314203,0.54369


# 5. Test Evaluation

Only the validation-selected rule is scored on test. Do not change the trial grid or
selection metric in response to this result.

In [10]:
test_result = evaluate_trial(final_model_name, "test")
display(
    test_result.select(
        "model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"
    )
)

model,split,mrr,ndcg_at_5,ndcg_at_10,recall_at_10
str,str,f64,f64,f64,f64
"""content_fusion_text_0.50""","""test""",0.278174,0.250449,0.310056,0.535561


# 6. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [11]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(
    example_user,
    score_file=OUTPUT_DIR / final_model_name / "validation_scores.parquet",
    feature_run=RUN,
    top_n=5,
)

user_code,impression_uid,news_code,score
i64,i64,i64,f64
11484,10154218,47020,0.786995
11484,10154218,12029,0.387898
